# 05 - Storytelling dos Resultados

Este notebook transforma os outputs finais da Aurora em uma narrativa curta para apresentação. Ele conecta dados, modelo, Power BI e impacto de negócio.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DADOS_RAW = ROOT / "dados" / "raw"
DADOS_PROCESSED = ROOT / "dados" / "processed"
DADOS_OUTPUTS = ROOT / "dados" / "outputs"
print(f"Raiz do projeto: {ROOT}")

import json

In [ ]:
with open(DADOS_OUTPUTS / "metricas_modelo.json", encoding="utf-8") as f:
    metricas = json.load(f)

predicoes = pd.read_csv(DADOS_OUTPUTS / "predicoes_churn.csv")
feature_importance = pd.read_csv(DADOS_OUTPUTS / "feature_importance.csv")
threshold_analysis = pd.read_csv(DADOS_OUTPUTS / "threshold_analysis.csv")
summary_path = DADOS_OUTPUTS / "summary.json"
summary = json.load(open(summary_path, encoding="utf-8")) if summary_path.exists() else {}

print("Arquivos carregados com sucesso.")

## 1. Resumo executivo do modelo

As métricas são geradas dinamicamente pelo pipeline. O README não fixa valores para evitar inconsistência entre execuções.

In [ ]:
metricas_principais = {k: metricas.get(k) for k in ["accuracy", "precision", "recall", "f1_score", "roc_auc", "baseline_churn_rate", "threshold_used"]}
display(pd.DataFrame(metricas_principais.items(), columns=["metrica", "valor"]))
print(metricas.get("recommended_threshold_reason", ""))

**Leitura para apresentação:** ROC-AUC mostra capacidade de ranking, precision mostra qualidade dos alertas e recall mostra capacidade de capturar clientes que poderiam sair.

## 2. Clientes prioritários para retenção

In [ ]:
cols = ["cliente_id", "nome", "estado", "perfil_risco", "prob_churn", "risco", "recomendacao"]
display(predicoes[cols].head(20))

risco = predicoes["risco"].value_counts().rename_axis("risco").reset_index(name="clientes")
display(risco)

**Leitura para apresentação:** a Aurora não decide automaticamente. Ela organiza uma fila de priorização para que a equipe humana de retenção aja com contexto.

## 3. Principais variáveis do modelo

In [ ]:
display(feature_importance.head(15))
feature_importance.head(10).sort_values("importance").plot(kind="barh", x="feature", y="importance", figsize=(8, 5), color="#8558f2", legend=False)
plt.title("Sinais mais importantes do modelo")
plt.tight_layout()
plt.show()

**Leitura para apresentação:** feature importance ajuda a explicar quais sinais sustentam o ranking de risco, conectando modelo e decisão de negócio.

## 4. Threshold analysis

A Aurora compara thresholds para retenção. Em churn, reduzir falso negativo costuma ser importante, mas sem criar alertas demais para a equipe operacional.

In [ ]:
display(threshold_analysis)

plt.figure(figsize=(7, 4))
plt.plot(threshold_analysis["threshold"], threshold_analysis["precision"], marker="o", label="precision")
plt.plot(threshold_analysis["threshold"], threshold_analysis["recall"], marker="o", label="recall")
plt.plot(threshold_analysis["threshold"], threshold_analysis["f1_score"], marker="o", label="f1_score")
plt.title("Comparação de thresholds")
plt.xlabel("Threshold")
plt.ylabel("Métrica")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Como conectar ao Power BI

Na apresentação, mostre as páginas nesta ordem:

1. **Visão Executiva:** KPIs principais, volume financeiro e distribuição de risco.
2. **Consumo e Comportamento Financeiro:** categorias, canais e matriz de perfil de risco.
3. **Churn e Retenção:** clientes prioritários e recomendações.
4. **Modelo ML:** métricas, feature importance e threshold analysis.
5. **Storytelling Executivo:** síntese do problema, solução, impacto e próximos passos.

In [ ]:
print("Resumo do dataset para fala:")
print("Clientes:", summary.get("total_clientes", "N/D"))
print("Transações:", summary.get("total_transacoes", "N/D"))
print("Fonte:", summary.get("data_source_name", "N/D"))
print("Transações sintéticas usadas:", summary.get("synthetic_transactions_used", "N/D"))

## 6. Roteiro curto para 5 minutos

**Problema:** empresas financeiras percebem tarde demais sinais de churn.

**Dados:** usamos o dataset público Churn Modelling do Kaggle como base de clientes e churn. Como ele não possui histórico transacional, criamos uma camada sintética reprodutível de transações.

**Análise:** exploramos renda, saldo, score, perfil de risco, estado, categorias e evolução mensal.

**Modelo:** treinamos Random Forest com class weight balanced, avaliando precision, recall, F1, ROC-AUC e thresholds.

**Dashboard:** o Power BI mostra visão executiva, consumo, churn, retenção e explicação do modelo.

**Impacto:** a Aurora prioriza clientes em risco para retenção consultiva, apoiando decisão humana com dados.

**Bônus:** o app React premium e a Análise Expressa mostram como a solução poderia virar produto estático, sem backend obrigatório.